In [39]:
import jsonlines
from pprint import pprint

```json
{
  "instruction": "你是一位苏格拉底式教师。学生类型：{student_type}。教学场景：{student_profile}。教学目标：{topic_text}",
  "input": "{历史对话拼接}\n学生：{最新学生发言}",
  "output": "<thought>\n【当前意图】：{teacher_intent}\n【教学策略】：{teaching_strategy}\n【学科转移】：{discipline_transfer}\n【判断】：{教师对学生认知状态的分析}\n</thought>\n{教师回复}"
}
```

In [40]:
def construct_system_prompt(data_item):
    """
    根据教学数据构建丰富的 System Prompt。
    将知识点(topic_text)和学生画像(student_type/scenario)注入系统提示词。
    """
    topic = data_item.get("topic_text", "通用生物教学")
    student_type = data_item.get("student_type", "普通学生")
    scenario = data_item.get("scenario", "")

    # 构建结构化的 System Prompt
    system_content = (
        "你是一位专业的教师。你的任务是根据给定的教学目标和知识点，"
        "采用启发式教学法引导学生进行思考。\n\n"
        f"### 教学背景与目标\n{topic}\n\n"
        f"### 当前学生画像\n类型：{student_type}\n场景描述：{scenario}\n\n"
        "### 教学要求\n"
        "1. 不要直接给出答案，通过提问引导学生。\n"
        "2. 结合学生的回答情况进行追问或评价。\n"
        "3. 最终引导学生达成教学目标。"
    )
    return system_content


In [41]:
def convert_to_qwen_format(raw_data_list):
    """
    将原始教学数据转换为 Qwen3-Instruct/ChatML 兼容的格式。
    处理了连续发言合并的问题。
    """
    sft_dataset = []

    for item in raw_data_list:
        # 1. 构建 System Message
        messages = [{"role": "system", "content": construct_system_prompt(item)}]

        # 2. 提取对话流
        # 优先使用 annotations，因为包含详细元数据，且顺序通常正确
        # 如果 annotations 不存在，回退到 dialogue
        source_dialogue = item.get("annotations", item.get("dialogue", []))

        if not source_dialogue:
            continue

        for turn in source_dialogue:
            # 角色映射
            role_map = {"学生": "user", "教师": "assistant"}

            # 获取原始角色 (可能是 'speaker' 或 'role' 字段)
            raw_role = turn.get("speaker", turn.get("role"))
            # 获取内容 (可能是 'utterance' 或 'content' 字段)
            content = turn.get("utterance", turn.get("content"))

            if not raw_role or not content:
                continue

            current_role = role_map.get(raw_role)
            if not current_role:
                continue  # 跳过未知角色

            # --- 关键逻辑：合并连续的相同角色发言 ---
            # Qwen/Llama 等模型训练通常要求 User/Assistant 严格交替
            # 你的数据末尾有连续3段教师发言，需要合并
            if len(messages) > 0 and messages[-1]["role"] == current_role:
                # 如果上一条消息的角色和当前一致，则合并内容
                # 添加换行符分隔
                messages[-1]["content"] += "\n\n" + content
            else:
                # 否则添加新消息
                messages.append({"role": current_role, "content": content})

        # 3. 只有当包含有效对话时才添加（去除只有system prompt的情况）
        if len(messages) > 1:
            sft_dataset.append({"conversations": messages})

    return sft_dataset


In [42]:
import os
# 遍历所有 annotated 下的 jsonl 文件
root_dir = "../datasets/annotated"
output_dir = "./work/work"
os.makedirs(output_dir, exist_ok=True)
all_formatted = []

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if file.endswith(".jsonl"):
            file_path = os.path.join(subdir, file)
            print(f"Processing: {file_path}")
            with jsonlines.open(file_path, "r") as f:
                data = [line for line in f]
            formatted_data = convert_to_qwen_format(data)
            all_formatted.extend(formatted_data)


Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_15.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_48.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_44.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_29.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_24.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_21.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_19.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_14.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_10.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_31.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_18.jsonl
Processing: ../datasets/annotated/SocraticLM/SocraticLM-interdis_topic_40.jsonl
Processing: ../datasets/annotated/Socrat

In [43]:
output_path = os.path.join(output_dir, "sft_data_qwen_format.jsonl")
with jsonlines.open(output_path, "w") as writer:
    writer.write_all(all_formatted)
print(f"全部处理完成，已保存到 {output_path}")

全部处理完成，已保存到 ./work/work/sft_data_qwen_format.jsonl
